In [1]:
# mount google drive
from google.colab import drive 
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Constants block
DATASET_NAME = "filtered_dataset_ferret_3"
ALIGNED_DATASET_PATH = f"/content/drive/MyDrive/Datasets_aligned/{DATASET_NAME}"
MORPHS_PATH = f"/content/drive/MyDrive/Morphs/{DATASET_NAME}"

In [12]:
%%bash
# Clonning elasticface and getting the pre-trained model checkpoint
# this one is bash script with strict exit on error

set -e

# clonning elasticface repo if it's not cloned already
if [ ! -d "ElasticFace" ]; then
    git clone "https://github.com/fdbtrs/ElasticFace"
fi

# installing gdown to get stuff from google drive
pip install -q gdown
# downloading the pre-trained model and doing some cleanup cause unfrotunately google drive just works that way
mkdir -p checkpoints
gdown 'https://drive.google.com/drive/folders/1q3ws_BQLmgXyiy2msvHummXq4pRqc1rx' --folder -O checkpoints
ls
# Clean up and rename
rm -f checkpoints/training.log
mv checkpoints/*.pth checkpoints/ElasticFaceArc.pth 2>/dev/null || true

echo "Setup completed successfully"

Processing file 1n0o1SGhtr_X7Y3scI6VFIUa6KW8aAQtb 295672backbone.pth
Processing file 1jGm6rHh-jJ40c34u5eXBgAhR3u4KHblH training.log
checkpoints
drive
ElasticFace
sample_data
Setup completed successfully


Retrieving folder contents
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=1n0o1SGhtr_X7Y3scI6VFIUa6KW8aAQtb
From (redirected): https://drive.google.com/uc?id=1n0o1SGhtr_X7Y3scI6VFIUa6KW8aAQtb&confirm=t&uuid=24d224a4-22b6-4fd1-9a77-5ec99b587f4b
To: /content/checkpoints/295672backbone.pth
100%|██████████| 261M/261M [00:02<00:00, 95.2MB/s] 
Downloading...
From: https://drive.google.com/uc?id=1jGm6rHh-jJ40c34u5eXBgAhR3u4KHblH
To: /content/checkpoints/training.log
100%|██████████| 788k/788k [00:00<00:00, 7.44MB/s]
Download completed


In [ ]:
# Loading the model 
import sys 
import os
import torch 
import re

# appending repo to hte path to be able to import code from it
sys.path.append("./ElasticFace")
print(os.listdir("."))
from backbones.iresnet import iresnet100 # imporing the iresnet 100 architecture from the repo


# selecting the device to run the model on
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# initializing the model and moving it to the selected device
model = iresnet100()
model.to(device)

# loading the checkpoint into the model architecture
checkpoint = torch.load("./checkpoints/ElasticFaceArc.pth", map_location=device)
model.load_state_dict(checkpoint)

# switching the model to evaluation mode
model.eval()

# testing if the model is working by passing a fake tensor through it
fake_input = torch.randn(1, 3, 112, 112).to(device)
with torch.no_grad():
    fake_output = model(fake_input)
print(f"Fake output shape: {fake_output.shape}, if that works, the model is loaded and working correctly.")

['.config', 'ElasticFace', 'checkpoints', 'drive', 'sample_data']
Using device: cuda


ValueError: Expected more than 1 value per channel when training, got input size torch.Size([1, 512])

In [ ]:
# get ALL images from the aligned dataset as a list of paths by walking its structure
image_files = sorted([
    os.path.join(root, f)
    for root, _, files in os.walk(ALIGNED_DATASET_PATH)
    for f in files if f.lower().endswith(('.png', '.jpg', '.jpeg'))
])

# Returns a list of dicts, one per morph:
# [{"morph": morph_path, "verification1": image_path, "verification2": image_path}, ...]
# The exact source probe names are extracted from the morph filename and excluded,
# while the remaining same-identity images are used as verification probes.
def get_evaluation_probes(morphs_path: str = MORPHS_PATH) -> list[dict]:
    results = []

    for morph in sorted(os.listdir(morphs_path)):
        if not morph.lower().endswith(('.png', '.jpg', '.jpeg')):
            continue

        # getting the full path to the morph and it's name without the extension
        morph_path = os.path.join(morphs_path, morph)
        morph_name = os.path.splitext(os.path.basename(morph))[0]

        # using regex, extracting the identities which were used to construct the morph
        identity_1_name = re.search(r"(?<=^morphed_).*(?=_and)", morph_name).group(0)
        identity_2_name = re.search(r"(?<=_and_).*(?=$)", morph_name).group(0)

        # getting identity prefixes to find other probes of the same identity
        identity_1_prefix = identity_1_name.split("_")[0]
        identity_2_prefix = identity_2_name.split("_")[0]

        # getting the paths to the other probes of the identities, excluding the morph identity itself
        identity_1_probes = sorted(
            file for file in image_files
            if os.path.basename(file).startswith(f"{identity_1_prefix}_")
            and os.path.splitext(os.path.basename(file))[0] != identity_1_name
        )
        identity_2_probes = sorted(
            file for file in image_files
            if os.path.basename(file).startswith(f"{identity_2_prefix}_")
            and os.path.splitext(os.path.basename(file))[0] != identity_2_name
        )

        if not identity_1_probes or not identity_2_probes:
            continue

        # getting first probe of each identity to use for verification
        results.append({
            "morph": morph_path,
            "verification1": identity_1_probes[0],
            "verification2": identity_2_probes[0],
        })

    return results


In [ ]:
# getting the evaluation probes for the morphs in the dataset
evaluation_probes = get_evaluation_probes(MORPHS_PATH)

In [ ]:
# Image tensor transformation pipeline to prepare the images for the model
from PIL import Image
from torchvision import transforms

transform_pipeline = transforms.Compose([
    transforms.Resize((112, 112)),
    transforms.ToTensor(), # going from [0, 255] to [0.0, 1.0]
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]) # normalizing to [-1, 1]
])

In [ ]:
# 2 funcitons which are used to determine how similar are the morphs to original people 
# first one gets the embedding of the image 
# second calculates the cosine similarity between two embeddings
def get_embedding(image_path: str) -> torch.Tensor:
    image = Image.open(image_path).convert("RGB")
    image_tensor = transform_pipeline(image).unsqueeze(0).to(device)

    with torch.no_grad():
        embedding = model(image_tensor)

    if isinstance(embedding, (tuple, list)):
        embedding = embedding[0]

    embedding = embedding.detach().cpu().float()

    if embedding.ndim > 1:
        embedding = embedding.reshape(embedding.shape[0], -1)
        if embedding.shape[0] == 1:
            embedding = embedding[0]

    return embedding


def cosine_similarity(a: torch.Tensor, b: torch.Tensor) -> float:
    a = a.view(-1)
    b = b.view(-1)
    denom = (torch.norm(a) * torch.norm(b)).clamp_min(1e-12)
    return float(torch.dot(a, b) / denom)

In [ ]:
import pandas as pd
import numpy as np

# make sure the model is in eval mode
model.eval()

CSV_OUTPUT_PATH = f"/content/drive/MyDrive/Morph_Evaluation/{DATASET_NAME}_cosine_similarity.csv"
os.makedirs(os.path.dirname(CSV_OUTPUT_PATH), exist_ok=True)


rows = []
for item in evaluation_probes:
    morph_path = item["morph"]
    verification1_path = item["verification1"]
    verification2_path = item["verification2"]

    if morph_path is None or verification1_path is None or verification2_path is None:
        continue

    morph_embedding = get_embedding(morph_path)
    verification1_embedding = get_embedding(verification1_path)
    verification2_embedding = get_embedding(verification2_path)

    sim1 = cosine_similarity(morph_embedding, verification1_embedding)
    sim2 = cosine_similarity(morph_embedding, verification2_embedding)

    rows.append({
        "morph": morph_path,
        "verification1": verification1_path,
        "verification2": verification2_path,
        "1_similarity": sim1,
        "2_similarity": sim2,
    })

results_df = pd.DataFrame(rows, columns=["morph", "verification1", "verification2", "1_similarity", "2_similarity"])
results_df.to_csv(CSV_OUTPUT_PATH, index=False)

print(f"Saved {len(results_df)} rows to: {CSV_OUTPUT_PATH}")
print(results_df.head())